# Milvus / watsonx.data: Conexión y schema

## 0. Instalación

In [ ]:
!pip install pymilvus --quiet

## 1. Credenciales
> El token lo encontrás en watsonx.data → Conexiones → Milvus.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

MILVUS_URI        = os.getenv("MILVUS_URI", "")
MILVUS_TOKEN     = os.getenv("MILVUS_TOKEN", "")
COLECCION            = os.getenv("COLECCION", "workshop_documentos")

print("Credenciales cargadas")

## 2. Conectar

In [ ]:
from pymilvus import connections, utility, Collection, MilvusClient

connections.connect(
    alias='default',
    uri=MILVUS_URI,
    token=MILVUS_TOKEN
)

print('Conectado a Milvus')

## 3. Ver colecciones disponibles

In [ ]:
colecciones = utility.list_collections()
print('Colecciones disponibles:')
for c in colecciones:
    marker = '  ← workshop' if c == COLECCION else ''
    print(f'  • {c}{marker}')

## 4. Explorar la colección del workshop

In [ ]:
col = Collection(COLECCION)
col.load()  # carga el índice en memoria

schema = col.schema
print(f'Colección: {COLECCION}')
print(f'Descripción: {schema.description}')
print()
print('Campos:')
print('-' * 50)
for field in schema.fields:
    es_pk = ' (primary key)' if field.is_primary else ''
    dims = f' [{field.params.get("dim", "")} dims]' if 'dim' in field.params else ''
    print(f'  {field.name:<20} {field.dtype.name}{dims}{es_pk}')

In [ ]:
# Estadísticas de la colección
stats = col.num_entities
print(f'\nEntidades (chunks) almacenadas: {stats}')

## 5. Búsqueda de similitud vectorial

In [ ]:
import random

# Dimensiones del embedding (debe coincidir con el campo vector del schema)
DIMS = 384  # ajustar según el modelo de embedding usado

# Vector de consulta de prueba (en producción viene del modelo de embedding)
query_vector = [random.uniform(-1, 1) for _ in range(DIMS)]

resultados = col.search(
    data=[query_vector],
    anns_field='embedding',       # campo vectorial del schema
    param={'metric_type': 'COSINE', 'ef': 64},
    limit=3,
    output_fields=['contenido', 'fuente']
)

print('Resultados de búsqueda vectorial (vector aleatorio de prueba):')
print('-' * 50)
for i, hit in enumerate(resultados[0], 1):
    print(f'\n[{i}] Distancia: {hit.distance:.4f}')
    print(f'    Fuente: {hit.entity.get("fuente", "N/A")}')
    contenido = hit.entity.get('contenido', '')[:150]
    print(f'    Contenido: {contenido}...')

print('\nCon un vector aleatorio los resultados no son semánticamente relevantes.')